# 01 – Data Preprocessing

Load the raw India weather/rainfall Excel file, inspect missing values, apply column-specific cleaning, and save `data/processed/clean_dataset.csv`.

**Tip:** For a one-command run of the full pipeline, use `python run_pipeline.py` from the project root.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = Path(".").resolve()

RAW_PATH = ROOT / "data" / "raw" / "india_weather_rainfall_data.xlsx"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Raw path exists:", RAW_PATH.exists())

In [ ]:
df = pd.read_excel(RAW_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
missing = df.isnull().sum()
missing_df = pd.DataFrame({
    "Missing Values": missing,
    "Percentage": (missing / len(df)) * 100,
}).sort_values("Percentage", ascending=False)
missing_df.to_csv(OUT_DIR / "missing_values_summary.csv")
missing_df

In [ ]:
# Parse dates and sort by station + date before interpolation
df["date_of_record"] = pd.to_datetime(df["date_of_record"])
df = df.sort_values(["station_name", "date_of_record"]).reset_index(drop=True)

before = len(df)
df = df.dropna(subset=["rainfall"]).reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows with missing rainfall; remaining {len(df):,}")

for col in ["min_temp", "max_temp"]:
    df[col] = df.groupby("station_name")[col].transform(
        lambda s: s.interpolate(method="linear", limit_direction="both")
    )
    df[col] = df.groupby("station_name")[col].transform(lambda s: s.fillna(s.median()))

for col in ["wind_speed", "air_pressure"]:
    df[col] = df.groupby("station_name")[col].transform(lambda s: s.fillna(s.median()))

for col in ["min_temp", "max_temp", "wind_speed", "air_pressure"]:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Missing after clean:")
print(df.isnull().sum())

In [ ]:
df = df.sort_values(["station_name", "date_of_record"]).reset_index(drop=True)
out = OUT_DIR / "clean_dataset.csv"
df.to_csv(out, index=False)
print(f"Saved {out} shape={df.shape}")
print(df.dtypes)